In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
project_path = '/content/drive/MyDrive/Apple_Dataset'
if os.path.exists(project_path):
    os.chdir(project_path)
    print(f"Đã truy cập: {os.getcwd()}")
else:
    print("Không tìm thấy thư mục. Kiểm tra lại tên folder trên Drive nhé!")

Đã truy cập: /content/drive/MyDrive/Apple_Dataset


In [ ]:
!ls /content/drive/MyDrive/Apple_Dataset/ultralytics

CITATION.cff	 examples	 README.md	  ultralytics.egg-info
CONTRIBUTING.md  LICENSE	 README.zh-CN.md  yolo26n.pt
docker		 mkdocs.yml	 tests
docs		 pyproject.toml  ultralytics


In [ ]:
%cd /content/drive/MyDrive/Apple_Dataset/ultralytics
!pip install -e .

/content/drive/MyDrive/Apple_Dataset/ultralytics
Obtaining file:///content/drive/MyDrive/Apple_Dataset/ultralytics
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.4.41-0.editable-py3-none-any.whl size=23305 sha256=9eeab2fe7941c8f73e596b4ac434bd849d8a40e7588b4245aa54b93431883b4d
  Stored in directory: /tmp/pip-ephem-wheel-cache-jlhx7f_g/wheels/f3/38/40/34061b2290a84783ebeb24f157aed4740fc0095421d61e4e00
Successfully built ultralytics
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.4.41
    Uninstalling ultralytics-8.4.41:
      Successfully uninstalled ultralytics-8.4.41


In [ ]:
import json
import os

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN TỪ GOOGLE DRIVE CỦA ÔNG
# ==========================================
COCO_JSON_PATH = '/content/drive/MyDrive/Apple_Dataset/data_v3/test/_annotations.json'
OUTPUT_DIR = '/content/drive/MyDrive/Apple_Dataset/data_v4/test'
# ==========================================

def convert_v3_to_v4_only_labels(json_path, output_dir):
    """
    Chỉ chuyển đổi nhãn COCO sang YOLO Polygon, KHÔNG đụng chạm đến ảnh gốc.
    """
    out_labels_dir = os.path.join(output_dir, 'labels')
    os.makedirs(out_labels_dir, exist_ok=True)

    print(f"Đang đọc dữ liệu từ: {json_path}")
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 1. BẢN ĐỒ CATEGORY
    category_mapping = {}
    for cat in data['categories']:
        if cat['name'] == 'Green_Apple':
            category_mapping[cat['id']] = 0
        elif cat['name'] == 'Red_Apple':
            category_mapping[cat['id']] = 1

    # 2. BẢN ĐỒ IMAGE
    image_mapping = {}
    for img in data['images']:
        image_mapping[img['id']] = {
            'file_name': img['file_name'],
            'width': img['width'],
            'height': img['height']
        }

    # 3. QUÉT NHÃN VÀ BIẾN HÌNH (BBOX -> POLYGON)
    total_annotations = 0
    pseudo_masks = 0
    processed_files = set() # Đếm số lượng file .txt thực tế được tạo

    for ann in data['annotations']:
        image_id = ann['image_id']
        category_id = ann['category_id']

        if category_id not in category_mapping:
            continue

        img_info = image_mapping[image_id]
        img_width = img_info['width']
        img_height = img_info['height']
        filename = img_info['file_name']
        txt_filename = os.path.splitext(filename)[0] + '.txt'
        yolo_class_id = category_mapping[category_id]

        # KIỂM TRA LÕI: Có Segmentation thật không? Nếu không, dùng Bbox chế tạo ra Segmentation giả
        if 'segmentation' in ann and len(ann['segmentation']) > 0 and len(ann['segmentation'][0]) > 4:
            poly = ann['segmentation'][0]
        else:
            bbox = ann['bbox'] # [x_min, y_min, width, height]
            x_min, y_min, w, h = bbox[0], bbox[1], bbox[2], bbox[3]
            poly = [
                x_min, y_min,             # Góc trên-trái
                x_min + w, y_min,         # Góc trên-phải
                x_min + w, y_min + h,     # Góc dưới-phải
                x_min, y_min + h          # Góc dưới-trái
            ]
            pseudo_masks += 1

        # Chuẩn hóa (Normalize) mảng polygon (Đưa về dải 0.0 - 1.0)
        normalized_polygon = []
        for i in range(0, len(poly), 2):
            x = max(0.0, min(1.0, poly[i] / img_width))
            y = max(0.0, min(1.0, poly[i+1] / img_height))
            normalized_polygon.extend([f"{x:.6f}", f"{y:.6f}"])

        # Ghi ra file .txt
        with open(os.path.join(out_labels_dir, txt_filename), 'a', encoding='utf-8') as txt_file:
            line = f"{yolo_class_id} " + " ".join(normalized_polygon) + "\n"
            txt_file.write(line)

        total_annotations += 1
        processed_files.add(txt_filename)

    print("-" * 40)
    print("HOÀN TẤT SINH NHÃN DATA_V4!")
    print(f"-> Đã tạo {len(processed_files)} file .txt chứa {total_annotations} nhãn.")
    print(f"-> Trong đó có {pseudo_masks} Bounding Box được bọc thành Polygon giả lập.")
    print(f"-> Kết quả lưu tại: {out_labels_dir}")
    print("\n* LƯU Ý: Đừng quên tự copy thư mục 'images' từ v3 sang v4 nhé!")

if __name__ == '__main__':
    convert_v3_to_v4_only_labels(COCO_JSON_PATH, OUTPUT_DIR)

Đang đọc dữ liệu từ: /content/drive/MyDrive/Apple_Dataset/data_v3/test/_annotations.json
----------------------------------------
HOÀN TẤT SINH NHÃN DATA_V4!
-> Đã tạo 100 file .txt chứa 6954 nhãn.
-> Trong đó có 0 Bounding Box được bọc thành Polygon giả lập.
-> Kết quả lưu tại: /content/drive/MyDrive/Apple_Dataset/data_v4/test/labels

* LƯU Ý: Đừng quên tự copy thư mục 'images' từ v3 sang v4 nhé!


In [ ]:
import torch
from ultralytics import RTDETR

# Cấu hình đường dẫn
MODEL_CONFIG = '/content/drive/MyDrive/Apple_Dataset/scripts/msrrt-detr-v2.yaml'
DATA_CONFIG = '/content/drive/MyDrive/Apple_Dataset/data.yaml' # Thay bằng file thật của ông
PROJECT_DIR = '/content/drive/MyDrive/Apple_Dataset/output'

def main():
    # Xóa cache GPU trước khi chạy
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # 1. Khởi tạo "Thể xác" MSRRT-DETR từ file thiết kế
    model = RTDETR(MODEL_CONFIG)

    # 2. Bơm "Linh hồn" (Tri thức nền tảng từ COCO, loại bỏ hoàn toàn rác của v3_final)
    # Hệ thống sẽ tự động tải file này về nếu chưa có
    model.load('rtdetr-l.pt')

    print("--- BẮT ĐẦU CHINH PHẠT MSRRT-DETR (KIẾN TRÚC CHUẨN) ---")

    # 3. Kích hoạt quá trình huấn luyện
    results = model.train(
        data=DATA_CONFIG,
        epochs=150,
        batch=16,                 # Colab Pro A100 cân tốt batch 16
        device=0,
        project=PROJECT_DIR,
        name='msrrt_detr_v5_mask_guided', # Đổi tên folder để không đụng chạm rác cũ

        # --- BỘ KHÓA CHỐNG OVERFIT (REGULARIZATION) ---
        dropout=0.05,
        weight_decay=0.001,

        # --- SIÊU VŨ KHÍ MASK-GUIDED ---
        copy_paste=0.5,
        # --- CHIẾN LƯỢC TĂNG CƯỜNG DỮ LIỆU (ONLINE AUGMENTATION) ---
        # A. Điều chỉnh không gian màu HSV
        hsv_h=0.015,              # Sắc độ (Hue)
        hsv_s=0.7,                # Độ bão hòa (Saturation)
        hsv_v=0.4,                # Độ sáng (Value)

        # B. Biến đổi hình học
        degrees=10.0,             # Xoay ngẫu nhiên
        translate=0.1,            # Tịnh tiến
        scale=0.5,                # Thu phóng
        fliplr=0.5,               # Lật ngang (50% tỷ lệ)
        flipud=0.0,               # Lật dọc

        # C. Mô phỏng che khuất (Occlusion & Complex Background)
        mosaic=0.5,               # Giảm Mosaic để bảo vệ tính toàn vẹn không gian
        mixup=0.1,                # 10% Mixup
        erasing=0.2,              # 20% Random Erasing

        # D. Tinh chỉnh giai đoạn cuối
        close_mosaic=25           # Đóng Mosaic ở 25 epoch cuối
    )

if __name__ == '__main__':
    main()

WARNING ⚠️ no model scale passed. Assuming scale='l'.
Transferred 11/1166 items from pretrained weights
--- BẮT ĐẦU CHINH PHẠT MSRRT-DETR (KIẾN TRÚC CHUẨN) ---
New https://pypi.org/project/ultralytics/8.4.46 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=25, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.5, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Apple_Dataset/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.05, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.2, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/150      31.6G      2.313     0.1577       1.73       1079        640: 100% ━━━━━━━━━━━━ 73/73 1.1it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 3.2it/s 2.2s
                   all        220       7652    0.00521     0.0485    0.00189   0.000418

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      2/150      32.8G      2.008     0.2566     0.5545       1254        640: 0% ──────────── 0/73  0.4s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/150      32.8G      1.666     0.3187      0.358       1062        640: 100% ━━━━━━━━━━━━ 73/73 1.4it/s 52.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 3.4it/s 2.1s
                   all        220       7652      0.732      0.229      0.201     0.0603

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      3/150      32.9G      1.611     0.3207     0.4325       1459        640: 0% ──────────── 0/73  0.4s

...

    148/150      38.3G     0.3468     0.4036    0.03039        484        640: 100% ━━━━━━━━━━━━ 73/73 2.4it/s 30.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 4.1it/s 1.7s
                   all        220       7652      0.847      0.853      0.908      0.668

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
    149/150      38.3G     0.3393     0.3884    0.03081        599        640: 0% ──────────── 0/73  0.4s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    149/150      38.3G      0.346     0.4013    0.02996        512        640: 100% ━━━━━━━━━━━━ 73/73 1.8it/s 41.7s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 3.9it/s 1.8s
                   all        220       7652      0.853      0.842      0.909      0.664

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
    150/150      38.3G     0.3271     0.3854    0.02465        917        640: 0% ──────────── 0/73  0.3s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    150/150      38.3G     0.3437     0.3988    0.02915        750        640: 100% ━━━━━━━━━━━━ 73/73 2.4it/s 30.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 4.0it/s 1.7s
                   all        220       7652      0.855      0.843      0.909      0.668

150 epochs completed in 2.345 hours.
Optimizer stripped from /content/drive/MyDrive/Apple_Dataset/output/msrrt_detr_v5_mask_guided-10/weights/last.pt, 130.5MB
Optimizer stripped from /content/drive/MyDrive/Apple_Dataset/output/msrrt_detr_v5_mask_guided-10/weights/best.pt, 130.5MB

Validating /content/drive/MyDrive/Apple_Dataset/output/msrrt_detr_v5_mask_guided-10/weights/best.pt...
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
msrrt-detr-v2 summary: 333 layers, 64,574,146 parameters, 0 gradients, 126.1 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━